In [1]:
import requests
import csv
import json
from dotenv import load_dotenv
import os


In [2]:

URL = "https://router.huggingface.co/v1/chat/completions"
load_dotenv()
API_TOKEN = os.getenv("HUGGING_FACE_API")
HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}
payload = {
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "messages": [
            {"role": "user", "content": "Classify sentiment: I love this product"}
        ],
        "max_tokens": 200
    }



In [3]:
try:
    response = requests.post(URL, headers=HEADERS, json=payload, timeout=30)
    print("Status:", response.status_code)

    if response.status_code == 200:
        r_data = response.json()
        print("Answer:", r_data["choices"][0]["message"]["content"])
    else:
        print("FAILED:", response.text)

except requests.exceptions.Timeout:
    print("Timed out!")
except Exception as e:
    print("Error!", e)

Status: 200
Answer: Positive sentiment.


In [4]:
# ─── SAFE REQUEST FUNCTION ────────────────────────────────
def make_request(messages, max_tokens=200):
    payload = {
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "messages": messages,
        "max_tokens": max_tokens
    }
    try:
        response = requests.post(URL, headers=HEADERS, json=payload, timeout=30)
        print("Status Code:", response.status_code)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        else:
            print("❌ Failed:", response.text)
            return None
    except requests.exceptions.Timeout:
        print("❌ Timed out!")
        return None
    except Exception as e:
        print("❌ Error:", e)
        return None

In [5]:
# ─── SENTIMENT ────────────────────────────────────────────
def sentiment_analysis(text, label):
    messages = [
        {
            "role": "system",
            "content": "You are a sentiment classifier. Given a text, respond with only one word: Positive, Negative, or Neutral."
        },
        {
            "role": "user",
            "content": f"Classify sentiment: {text}"
        }
    ]

    result = make_request(messages, max_tokens=10)

    print(f"\n{'='*60}")
    print(f"Sentiment — {label}")
    print(f"Input:  {text}")
    print(f"Output: {result if result else '❌ Failed'}")
    print("="*60)

In [6]:
# ─── SUMMARIZATION ────────────────────────────────────────
def summarize(text, label):
    messages = [
        {
            "role": "system",
            "content": "You are a summarization assistant. Summarize the given text concisely in 1-2 sentences."
        },
        {
            "role": "user",
            "content": f"Summarize: {text}"
        }
    ]

    result = make_request(messages, max_tokens=80)

    print(f"\n{'='*60}")
    print(f"Summarization — {label}")
    print(f"Input:  {text}")
    print(f"Output: {result if result else '❌ Failed'}")
    print("="*60)

In [11]:
# ─── TEST ─────────────────────────────────────────────────

sentiment_analysis(
    "I absolutely love this product! It works perfectly.",
    "Positive"
)

sentiment_analysis(
    "I am not happy i did not complete my tasks",
    "Negative"
)

summarize(
    "422 Error",
    "Short Text"
)

print("\n✅ Done")

Status Code: 200

Sentiment — Positive
Input:  I absolutely love this product! It works perfectly.
Output: Positive
Status Code: 200

Sentiment — Negative
Input:  I am not happy i did not complete my tasks
Output: Negative
Status Code: 200

Summarization — Short Text
Input:  422 Error
Output: A 422 Unprocessable Entity error is an HTTP status code indicating that the server understands the request but was unable to process it due to a client-side error, often related to validation issues. This error typically occurs when the server cannot complete the request as the data is invalid or incomplete.

✅ Done
